In [ ]:
import obi_auth
import numpy
import pandas
import neurom

from obi_notebook.get_projects import get_projects
from obi_notebook.get_entities import get_entities

from entitysdk.models import  Circuit
from entitysdk import Client, models, types

from conntility import ConnectivityMatrix
from scipy.sparse import coo_matrix

token = obi_auth.get_token(environment="staging", auth_mode="daf")
project_context = get_projects(token, env="staging")


# Setup

To begin, we create a ConnectivityMatrix object that specifies the circuit we want to build.
As a ConnectivityMatrix, synaptic connections are natively specified.

Additionally, we specify the MEModels of the neurons to simulate in the form of vertex properties. Specifically, the IDs of the MEModel entities are encoded in string form.

## Find MEModels and create target connectivity matrix

Here, we simply find the first five MEModel entities with CalibrationResults. For production, something more complex would be done.

For connectivity, we wire up a directed simplex.

In [ ]:
from obi_one.scientific.library.wire_circuit.node_pop_from_matrix import COL_ME_MODEL_ID_

entity_client = Client(token_manager=token, project_context=project_context, environment="staging")

# Find random MEModels
results_iter = entity_client.search_entity(
    entity_type=models.MEModel
)
memodel_ids = []
for _ in range(10):
    results_iter.__next__()
while len(memodel_ids) < 5:
    cand_mdl = results_iter.__next__()
    if cand_mdl.calibration_result is not None:
        memodel_ids.append(str(cand_mdl.id))

# Target connectivity
# Integers specify the number of synapses between a pair
matrix = numpy.array([
    [0, 0, 3, 2, 1],
    [0, 0, 0, 3, 2],
    [3, 0, 0, 0, 3],
    [0, 4, 0, 0, 0],
    [4, 0, 3, 0, 0]
])
# MEModel IDs specified as vertex properties
nodes = pandas.DataFrame({
    COL_ME_MODEL_ID_ : memodel_ids
})
edges = pandas.concat([pandas.DataFrame({
    "row": coo_matrix(numpy.maximum(matrix - n_, 0)).row,
    "col": coo_matrix(numpy.maximum(matrix - n_, 0)).col,
    "data": numpy.ones(coo_matrix(numpy.maximum(matrix - n_, 0)).nnz)
}) for n_ in range(matrix.max())], axis=0).reset_index(drop=True)

# ConnectivityMatrix objet serves as "recipe"
M = ConnectivityMatrix(edges[["row", "col"]], edge_properties=edges[["data"]],
                       vertex_properties=nodes, shape=matrix.shape)


# Parameterize morphology locations

Additional edge properties determine how the neurite locations of synapses are determined.

For now, exponential profiles on the path distance from the soma are implemented and adjustment factors for section types.
For each edge we can separately parameterize these aspects.

  - column: "exponential_scale": probability that a morphology location is picked is proportional to exp(-soma_path_distance * exponential_scale) for positive values of exponential_scale. That is: More likely towards the soma. For negative values it is proportional to 1.0 - exp(soma_path_distance * exponential_scale). That is: Less likely towards the soma.
  Default if not specified is 0.0, i.e. uniform probability.
  - column: AXON: scales probabilities for axonal segments. Default: 0.0, i.e. no synapses on the axon
  - column: APICAL_DENDRITE: scales probabilities for apical segments. Default: 1.0
  - column: BASAL_DENDRITE: scales probabilities for basal segments. Default: 1.0

In [ ]:
# Example: We specify different exponental scaling strategies for different pre-synaptic neurons
# To implement that, we create a Series with different values for each me_model_id
tgt_path_distance_scale = pandas.Series(dict(zip(memodel_ids, [0.01, 0.001, 0.0, -0.01, -0.001])))
# Returns the me_model_id of the pre-synaptic neuron of each edge
presyn_me_model = M.edge_associated_vertex_properties(COL_ME_MODEL_ID_)["row"]
# Set value accordingly
M.add_edge_property("exponential_scale", tgt_path_distance_scale[presyn_me_model].to_numpy())
# Axon synapses possible, but unlikely
M.add_edge_property(str(neurom.AXON), [0.05] * len(M.edges))
# Apical synapses 10 times more likely
M.add_edge_property(str(neurom.APICAL_DENDRITE), [0.5] * len(M.edges))
# Basal synapses even more likely
M.add_edge_property(str(neurom.BASAL_DENDRITE), [1.0] * len(M.edges))

## Create circuit_connectivity_matrices asset

Next, we create a folder holding the created ConnectivityMatrix and a table-of-contents json file.
This is the format used in entitycore for the ConnectivityMatrices of Circuit objects.

That is, usually, a Circuit is created in SONATA format and then a ConnectivityMatrix extracted from it. Here, we invert that by first specifying the matrix and then building the SONATA according to it.

In [ ]:
import os
import json

root = "circuit_connectivity_matrices"
sub_root = "recipe"
os.makedirs(os.path.join(root, sub_root), exist_ok=True)

matrix_path = os.path.join(root, sub_root, "matrix.h5")
M.to_h5(matrix_path)
toc = {
    "recipe": {
        "single": {
            "description": "Recipe defining the MEmodels and connectivity",
            "path": os.path.join(sub_root, "matrix.h5")
        }
    }
}
toc_path = os.path.join(root, "matrix_config.json")
with open(toc_path, "w") as fid:
    json.dump(toc, fid, indent=2)


## Register the Circuit and its assets
We register a new Circuit object and upload the ConnectivityMatrix as an asset

In [ ]:
# Random region and subject. This is just a test!
region = entity_client.search_entity(
    entity_type=models.BrainRegion
).first()
subject = entity_client.search_entity(
    entity_type=models.Subject,
    query={"name__ilike": "Generic"}
).first()

# UPLOAD!
circ_entity = Circuit(
    name="Test wire-up circuit #1",
    description="Test of the workflow to wire a custom SONATA circuit",
    number_neurons=len(M),
    number_connections=M.matrix.tocsc().nnz,
    number_synapses=len(M.edges),
    scale=types.CircuitScale.small,
    build_category=types.CircuitBuildCategory.computational_model,
    brain_region=region,
    subject=subject
)
registered_entity = entity_client.register_entity(
    entity=circ_entity
)

entity_client.upload_directory(
    entity_id=registered_entity.id,
    entity_type=Circuit,
    name="ConnectivityMatrix",
    paths={
        "matrix_config.json": toc_path,
        os.path.join(sub_root, "matrix.h5"): matrix_path
    },
    label=types.AssetLabel.circuit_connectivity_matrices,
)

# Wire structural SONATA circuit task

Finally, we can create and execute a task to create a SONATA circuit according to that matrix.

Note two things:
  - At this moment, the created circuit will not be registered as an asset to the Circuit.
  - Anatomical synapse locations are not created. An edge file is created, but it is barebones, specifying only pre- and post-synaptic neurons and no further properties

Both issues are to be fixed soon.

In [ ]:
from obi_one.scientific.tasks.wire_circuit.task import WireStructuralCircuitTask, WireStructuralCircuitSingleConfig
from obi_one.core.info import Info
from obi_one.scientific.from_id.circuit_from_id import CircuitFromID

OUTPUT_DIR = "wired_SONATA"

task_cfg = WireStructuralCircuitSingleConfig(
    info=Info(
        campaign_name="Test wire-up #1",
        campaign_description="Testing the wire-up task"
    ),
    initialize=WireStructuralCircuitSingleConfig.Initialize(
        circuit=CircuitFromID(id_str=str(registered_entity.id))
    ),
    coordinate_output_root=OUTPUT_DIR
)
task = WireStructuralCircuitTask(
    config=task_cfg
)
task.execute(db_client=entity_client)

# Cleanup

Delete the entity...

In [ ]:
entity_client.delete_entity(entity_id=registered_entity.id, entity_type=Circuit)